In [ ]:
#Parameters
# --- Imports ---
import yfinance as yf
import pandas as pd
import yfinance as yf
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- Parameters ---
ticker = "SPY"
period = "max"
n = 14 # days ahead to predict
train_ratio = 0.8



In [ ]:
#Functions
# --- Label Generator ---
def future_positive_return_labels(close_prices, n=1):
    """
    Generate binary labels for whether the price is higher n days later,
    but mark NaN where future data is unavailable.
    """
    # Price n days ahead
    future_close = close_prices.shift(-n)
    
    # % return from today to n days later
    future_return = (future_close - close_prices) / close_prices
    
    # 1 if positive return, 0 if not
    labels = (future_return > 0).astype("float")  # float so we can have NaN
    
    # Explicitly set last n rows to NaN (no future data)
    labels.iloc[-n:] = float("nan")
    
    return labels


def create_lagged_returns(return_series, lags):
    """
    Create lagged return features from a return series.
    
    Parameters:
    - return_series: pd.Series of returns (e.g., daily returns)
    - lags: list of integers representing lag days
    
    Returns:
    - pd.DataFrame with lagged return columns named 'return_lag_{lag}'
    """
    df = pd.DataFrame(index=return_series.index)
    for lag in lags:
        df[f'return_lag_{lag}'] = return_series.shift(lag)
    return df

def moving_average_regime(close, short_window=50, long_window=200):
    """
    Detect regime based on moving average crossover.
    
    Returns a binary Series: 1 = uptrend (short SMA > long SMA), 0 = downtrend.
    """
    sma_short = close.rolling(window=short_window).mean()
    sma_long = close.rolling(window=long_window).mean()
    regime = (sma_short > sma_long).astype(int)
    return regime

def rsi(close, window=14):
    """
    Compute the Relative Strength Index (RSI) for a given window.
    """
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def create_features_improved(open_, high, low, close, volume):
    df = pd.DataFrame(index=close.index)
    
    periods = range(1,50) # 1 for daily returns, plus others

    for period in periods:
        if period == 1:
            df['return'] = close.pct_change()
        else:
            df[f'return_{period}'] = close.pct_change(period)
        
    # Daily price ranges
    df['high_low_range'] = (high - low) / close
    df['close_open_diff'] = (close - open_) / open_
    
    
    # Volume related features
    df['volume'] = volume
    df['volume_change'] = volume.pct_change()
    df['volume_change_5'] = volume.pct_change(5)
    df['volume_ma_5'] = volume.rolling(window=5).mean()
    df['volume_ma_10'] = volume.rolling(window=10).mean()
    df['volume_ma_20'] = volume.rolling(window=20).mean()

    
    
    # Lagged returns
    lags = range(1, n)
    lagged_returns_df = create_lagged_returns(df['return'], lags)
    df = pd.concat([df, lagged_returns_df], axis=1)
    
    # Regime features
    df['trend_regime_5_21_crossover'] = moving_average_regime(close, short_window=5, long_window=21)
    df['trend_regime_21_50_crossover'] = moving_average_regime(close, short_window=21, long_window=50)
    df['trend_regime_50_200_crossover'] = moving_average_regime(close)
    
    # Time features
    df['day_of_week'] = close.index.dayofweek
    df['day_of_month'] = close.index.day
    df['week_of_year'] = close.index.isocalendar().week
    df['year'] = close.index.year
    df['quarter'] = close.index.quarter
    df['week_of_month'] = (close.index.day - 1) // 7 + 1
    df['month'] = close.index.month
    for window in range(1, 51):
        df[f'rsi_{window}'] = rsi(close, window=window)
        
    # Shift all features forward by n days so features at time t predict returns at t + n
    df = df.shift(n)
    
    df = df.dropna()
    return df

# Baseline features: lagged returns only
def create_features_baseline(close_prices):
    df = pd.DataFrame({'close': close_prices})
    for lag in range(1, 4):
        df[f'return_lag_{lag}'] = close_prices.pct_change(lag)
    df = df.dropna()
    return df

In [ ]:
#Feature Engineering and Label Creation

data = yf.download(ticker, period=period, auto_adjust=True)

# Extract OHLCV
open_prices = data['Open']
close_prices = data['Close']
high = data['High']
low = data['Low']
volume = data['Volume']

# Business day reindex with forward fill
full_range = pd.date_range(start=close_prices.index.min(), end=close_prices.index.max(), freq='B')
open_prices_bdays = open_prices.reindex(full_range).ffill()
close_prices_bdays = close_prices.reindex(full_range).ffill()
high_bdays = high.reindex(full_range).ffill()
low_bdays = low.reindex(full_range).ffill()
volume_bdays = volume.reindex(full_range).ffill()

def ensure_series(x):
    if isinstance(x, pd.DataFrame):
        return x.iloc[:, 0]
    else:
        return x

close_prices_series = ensure_series(close_prices_bdays)
high_series = ensure_series(high_bdays)
low_series = ensure_series(low_bdays)
open_series = ensure_series(open_prices_bdays)
volume_series = ensure_series(volume_bdays)

# Create features
features_improved = create_features_improved(close_prices_series, high_series, low_series, open_series, volume_series)
features_baseline = create_features_baseline(close_prices_series)

# 1. Create labels (with NaNs for missing future data)
labels_n_days = future_positive_return_labels(close_prices_series, n=n)

# 2. Drop NaN labels upfront (removes last n rows with no future price)
labels_n_days = labels_n_days.dropna()
print(labels_n_days.value_counts(normalize=True))
# ---- FIX FOR IMPROVED FEATURES ----
# Instead of dropna(), slice from first valid index to keep valid rows after lag windows
first_valid_idx = features_improved.first_valid_index()
features_improved_cleaned = features_improved.loc[first_valid_idx:]

# Now align features and labels for improved features
common_index_improved = features_improved_cleaned.index.intersection(labels_n_days.index)
features_improved_aligned = features_improved_cleaned.loc[common_index_improved]
labels_improved_aligned = labels_n_days.loc[common_index_improved]
#make sure its a pandas series


# For baseline features, just drop NaNs as usual (likely few NaNs)
features_baseline_clean = features_baseline.dropna()
common_index_baseline = features_baseline_clean.index.intersection(labels_n_days.index)
features_baseline_aligned = features_baseline_clean.loc[common_index_baseline]
labels_baseline_aligned = labels_n_days.loc[common_index_baseline]

In [ ]:
#Split data
split_idx_improved = int(len(labels_improved_aligned) * train_ratio)
X_train_improved = features_improved_aligned.iloc[:split_idx_improved]
X_test_improved = features_improved_aligned.iloc[split_idx_improved:]
y_train_improved = labels_improved_aligned.iloc[:split_idx_improved]
y_test_improved = labels_improved_aligned.iloc[split_idx_improved:]

split_idx_baseline = int(len(labels_baseline_aligned) * train_ratio)
X_train_baseline = features_baseline_aligned.iloc[:split_idx_baseline]
X_test_baseline = features_baseline_aligned.iloc[split_idx_baseline:]
y_train_baseline = labels_baseline_aligned.iloc[:split_idx_baseline]
y_test_baseline = labels_baseline_aligned.iloc[split_idx_baseline:]

print(f"Training samples improved: {len(X_train_improved)}")
print(f"Training samples baseline: {len(X_train_baseline)}")

print(f"NaNs in improved features after slicing from first valid index: {features_improved_aligned.isna().sum().sum()}")
print(f"Shape of improved features after slicing and alignment: {features_improved_aligned.shape}")

print(f"Length of labels_n_days after dropna: {len(labels_n_days)}")
print(f"Length of features_improved before cleaning: {len(features_improved)}")
print(f"Length of features_improved after cleaning: {len(features_improved_cleaned)}")
print(f"Length of features_improved after alignment: {len(features_improved_aligned)}")

print(f"First 5 indices of labels: {labels_n_days.index[:5]}")
print(f"First 5 indices of features_improved: {features_improved.index[:5]}")
print(f"First 5 indices of features_improved_aligned: {features_improved_aligned.index[:5]}")

display(features_improved.head(10))
display(features_improved.tail(10))  

In [ ]:
#Feature filtering
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance

# --- Check training data non-empty ---
if len(X_train_improved) == 0 or len(y_train_improved) == 0:
    raise ValueError("Training data for improved features is empty! Please check your feature creation or data alignment.")

if len(X_train_baseline) == 0 or len(y_train_baseline) == 0:
    raise ValueError("Training data for baseline features is empty! Please check your feature creation or data alignment.")

# --- Train models ---
model_improved = RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
model_improved.fit(X_train_improved, y_train_improved)

model_baseline = RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
model_baseline.fit(X_train_baseline, y_train_baseline)

# --- Predict on test sets ---
y_pred_improved = model_improved.predict(X_test_improved)
y_pred_baseline = model_baseline.predict(X_test_baseline)

# (Optional) Evaluate accuracy
print("Test accuracy with improved features:", accuracy_score(y_test_improved, y_pred_improved))
print("Test accuracy with baseline features:", accuracy_score(y_test_baseline, y_pred_baseline))

# --- Permutation importance for improved features ---
result_improved = permutation_importance(model_improved, X_test_improved, y_test_improved, n_repeats=10, random_state=42)
perm_importance_improved_df = pd.DataFrame({
    'feature': X_test_improved.columns,
    'importance_mean': result_improved.importances_mean,
    'importance_std': result_improved.importances_std
}).sort_values(by='importance_mean', ascending=False)

# --- Step 1: Tree-based feature importance ---
rf_importances = pd.Series(model_improved.feature_importances_, index=X_train_improved.columns)
tree_selected_features = rf_importances[rf_importances > 0.01].index.tolist()
print(f"Tree-based selected features: {len(tree_selected_features)}")

# --- Step 2: Permutation importance ---
perm_selected_features = perm_importance_improved_df[
    perm_importance_improved_df['importance_mean'] > 0
]['feature'].tolist()
print(f"Permutation importance selected features: {len(perm_selected_features)}")

# --- Step 3: Univariate feature selection ---
selector_uni = SelectKBest(mutual_info_classif, k=30)  # adjust k as needed
selector_uni.fit(X_train_improved.fillna(0), y_train_improved)  # Impute missing values
uni_selected_features = X_train_improved.columns[selector_uni.get_support()].tolist()
print(f"Univariate selected features: {len(uni_selected_features)}")

# --- Step 4: Combine all selected features ---
combined_features = list(set(tree_selected_features) | set(perm_selected_features) | set(uni_selected_features))
print(f"Combined features before correlation filtering: {len(combined_features)}")

# --- Step 5: Correlation filtering ---
def correlation_filter(df, features, threshold=0.9):
    selected = []
    corr_matrix = df[features].corr().abs()
    to_drop = set()
    for i in range(len(corr_matrix.columns)):
        feature = corr_matrix.columns[i]
        if feature in to_drop:
            continue
        selected.append(feature)
        correlated = corr_matrix.columns[(corr_matrix.iloc[i] > threshold)].tolist()
        if feature in correlated:
            correlated.remove(feature)
        to_drop.update(correlated)
    return selected

filtered_features = correlation_filter(X_train_improved, combined_features, threshold=0.9)
print(f"Features after correlation filtering: {len(filtered_features)}")

# --- Optional Step 6: RFE refinement ---
from sklearn.feature_selection import RFE
estimator = RandomForestClassifier(n_estimators=100, random_state=42)
selector = RFE(estimator, n_features_to_select=20, step=1)
selector.fit(X_train_improved[filtered_features].fillna(0), y_train_improved)
final_features = list(X_train_improved[filtered_features].columns[selector.support_])
print(f"Features after RFE selection: {len(final_features)}")

# --- Final feature set ---
final_features = filtered_features  # or replace with RFE features if used
print("Final selected features:", final_features)


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Prepare training and test data with final features
X_train_final = X_train_improved[final_features].fillna(0)
X_test_final = X_test_improved[final_features].fillna(0)

# Initialize and train the model
model_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
model_final.fit(X_train_final, y_train_improved)

# Predict probabilities instead of class labels
y_proba_final = model_final.predict_proba(X_test_final)[:, 1]

# Sweep thresholds
thresholds = np.arange(0.5, 0.91, 0.05)

# Initialize lists for plotting
accuracies = []
precisions = []
recalls = []
f1_scores = []

best_f1 = -1
best_f1_threshold = None
best_f1_predictions = None

best_prec = -1
best_prec_threshold = None
best_prec_predictions = None

print("Threshold | Accuracy | Precision | Recall | F1 Score")
print("----------------------------------------------------")

for threshold in thresholds:
    y_pred = (y_proba_final >= threshold).astype(int)
    acc = accuracy_score(y_test_improved, y_pred)
    prec = precision_score(y_test_improved, y_pred, zero_division=0)
    rec = recall_score(y_test_improved, y_pred, zero_division=0)
    f1 = f1_score(y_test_improved, y_pred, zero_division=0)

    # Append to lists for plotting
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)

    print(f"{threshold:.2f}     | {acc:.4f}   | {prec:.4f}   | {rec:.4f}   | {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_f1_threshold = threshold
        best_f1_predictions = y_pred.copy()

    if prec > best_prec:
        best_prec = prec
        best_prec_threshold = threshold
        best_prec_predictions = y_pred.copy()

print(f"\nBest F1 threshold: {best_f1_threshold:.2f} with F1 {best_f1:.4f}")
print(f"Best Precision threshold: {best_prec_threshold:.2f} with Precision {best_prec:.4f}")

# Plot the threshold, accuracy, precision, recall and f1 score
figure = go.Figure()

# Add traces for each metric
figure.add_trace(go.Scatter(x=thresholds, y=accuracies, mode='lines+markers', name='Accuracy'))
figure.add_trace(go.Scatter(x=thresholds, y=precisions, mode='lines+markers', name='Precision'))
figure.add_trace(go.Scatter(x=thresholds, y=recalls, mode='lines+markers', name='Recall'))
figure.add_trace(go.Scatter(x=thresholds, y=f1_scores, mode='lines+markers', name='F1 Score'))

# Update layout
figure.update_layout(
    title='Model Performance Metrics vs. Classification Threshold',
    xaxis_title='Classification Threshold',
    yaxis_title='Score',
    legend_title='Metrics',
    template='plotly_white'
)

figure.show()


In [ ]:

def plot_actual_vs_predicted(dates, actual, predicted, title):
    plot_df = pd.DataFrame({
        'Date': dates,
        'Actual': actual,
        'Predicted': predicted
    })

    fig = go.Figure()

    # Actual bars
    fig.add_trace(go.Bar(
        x=plot_df['Date'],
        y=plot_df['Actual'],
        name='Actual',
        marker_color='blue',
        opacity=0.7,
        hovertemplate='Date: %{x}<br>Actual: %{y}<extra></extra>'
    ))

    # Predicted dots
    fig.add_trace(go.Scatter(
        x=plot_df['Date'],
        y=plot_df['Predicted'],
        mode='markers',
        name='Predicted',
        marker=dict(color='red', size=8, symbol='circle'),
        hovertemplate='Date: %{x}<br>Predicted: %{y}<extra></extra>'
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Date',
        yaxis_title='Label',
        yaxis=dict(tickvals=[0, 1], ticktext=['0 (Negative)', '1 (Positive)']),
        height=500,
        legend=dict(y=0.99, x=0.01)
    )
    fig.show()

# Plot for best F1 threshold
plot_actual_vs_predicted(
    y_test_improved.index,
    y_test_improved,
    best_f1_predictions,
    f"Actual vs Predicted Labels (Best F1 = {best_f1:.4f} at threshold {best_f1_threshold:.2f})"
)

# Plot for best Precision threshold
plot_actual_vs_predicted(
    y_test_improved.index,
    y_test_improved,
    best_prec_predictions,
    f"Actual vs Predicted Labels (Best Precision = {best_prec:.4f} at threshold {best_prec_threshold:.2f})"
)
